In [1]:
from langchain_core.documents import Document

# Ingestion Pipeline

### Data to Documents

In [2]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

/tmp/ipykernel_274368/220481367.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader
/home/saphal/anaconda3/envs/ml_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_all_pdf():
    folder_path = "data/pdf"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path): #get name of all docs in folder
        if filename.lower().endswith(".pdf"):
            #complete file path
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            document = loader.load()
            
            all_docs.extend(document)
            num_docs += 1
    print("total pdfs:",num_docs)
    print("total pages:",len(all_docs))
    return all_docs

In [4]:
all_pdf_documents = load_all_pdf()

total pdfs: 2
total pages: 32


In [5]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

# Chunking

In [6]:
# !pip install langchain_text_splitters

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents,chunk_size=500,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [8]:
chunks = split_doc(all_pdf_documents)

In [9]:
len(chunks)

321

# Embeddings

In [10]:
from sentence_transformers import SentenceTransformer

In [11]:
class EmbeddingManager: #embedding model
    def __init__(self,model_name = "all-MiniLM-L6-v2"):
        
        self.model_name = model_name
        print("loading model....:",self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimension:",self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar = True)
        print("embeddings shape:",embeddings.shape)
        return embeddings

In [12]:
embedding_manager = EmbeddingManager()

loading model....: all-MiniLM-L6-v2


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3206.68it/s]


embedding dimension: 384


/tmp/ipykernel_274368/357030263.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimension:",self.model.get_sentence_embedding_dimension())


# Vector store

In [13]:
import chromadb
import uuid #create index for values

In [27]:
class VectorStoreManager:
    def __init__(self,persist_directory="data/vector_store",collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok=True)

        #create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata={"description":"This is a vector store collection for pdf embeddings in RAG"}
        )

        print("initalized the vector store with collection:",self.collection_name)
        print("docs in collection:",self.collection.count())

    #to add all docs in vector store
    def add_documents(self,documents,embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        #store => ids, embedding, document , metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (document,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4()}" #generate unique id
            ids.append(doc_id)
    
            metadata = dict(document.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(document.page_content)
            all_metadata.append(metadata)
    
            documents_content.append(document.page_content)
            embeddings_list.append(embedding.tolist())
    
            self.collection.add(
                ids=ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )

        print("total docs added in collection=", len(documents_content))
        print("docs in collection:",self.collection.count())

In [28]:
vector_store = VectorStoreManager()

initalized the vector store with collection: pdf_documents
docs in collection: 642


In [29]:
#chunks to embedding

texts = [document.page_content for document in chunks]

embedding = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks,embedding)

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:10<00:00,  1.08it/s]


embeddings shape: (321, 384)
total docs added in collection= 321
docs in collection: 963


# Retrieval Pipeline

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self,query,top_k=5,score_threshold=0.0):
        #query to embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        #retrieve value from vector store (semantic search)
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        #cosine similarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i,(doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score = 1-distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank": i+1
                    })
            
            print(f"retrieved {len(retrieved_docs)} documents")

        else:
             print("no docs found")

        return retrieved_docs

In [35]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)

In [36]:
rag_retriever.retrieve("What is RAG ")

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 11.53it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_691cac7d-0685-488c-af9b-5a620ae185ee',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'page_label': '1',
   'title': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'producer': 'pdfTeX-1.40.25',
   'total_pages': 21,
   'doc_index': 88,
   'subject': '',
   'author': '',
   'keywords': '',
   'creator': 'LaTeX with hyperref',
   'source': 'data/pdf/rag_for_llm.pdf',
   'content_length': 288,
   'trapped': '/False',
   'page': 0,
   'creationdate': '2024-03-28T00:54:45+00:00'},
  'distance': 0.4226759374141693,
  'similarity_score': 0.5773240625858307,
  'rank': 1},
 {'id': 'doc_e37d3727-